# Aula 26: Deploy e MLOps (Lab)

Neste Lab você põe um app de previsão **no ar**, com um endereço que qualquer pessoa pode abrir, e
depois vê o app se **refazer sozinho** quando o código muda no GitHub.

O app estima o preço de venda de uma casa em Ames (Iowa, EUA). O modelo é o pipeline da a11: as
mesmas 5 colunas numéricas e 3 categóricas, o `ColumnTransformer` (preencher, escalar, codificar) e a
`Ridge`, treinado com as 1.460 casas do Ames e salvo num arquivo.

| passo | o que você faz | onde |
|---|---|---|
| **L1** | exporta o modelo, baixa os arquivos e olha o `app.py` | Colab, EAD e seu computador |
| **L2** | cria um repositório e sobe os 4 arquivos | GitHub |
| **L3** | põe o app no ar | Streamlit Community Cloud |
| **L4** | muda o código e vê o app se refazer (auto-deploy) | GitHub e o app |
| **L5** | consulta, se algo der errado | o app e os logs |
| **L6** | *(opcional)* leva o modelo do seu projeto para um app | Colab, GitHub e Streamlit |

> 🧭 **Só o L1 tem código**, para rodar aqui no Colab. Do L2 em diante, este notebook é um guia:
> deixe-o aberto ao lado do navegador e siga os passos em ordem.

> ✅ **Você vai precisar de** uma conta no **GitHub** (se não tem, crie em https://github.com/signup) e
> dos arquivos do app, que vêm no zip da aula, no EAD, numa pasta `lab`. O Streamlit Community Cloud é gratuito e entra com a
> conta do GitHub.

---
## L1 · Exportar o modelo e baixar os arquivos

### Treinar e salvar o modelo
O app usa um modelo salvo num arquivo, e quem gera esse arquivo é você, aqui no Colab. Rode as três
células abaixo, em ordem.

A primeira célula monta o pipeline da a11 e o treina com as 1.460 casas do Ames. O treino usa todas
as casas, porque a avaliação do modelo já foi feita nas aulas de validação.

In [1]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

AMES = "https://www.inf.puc-rio.br/~abranco/files/datasets/house_prices.csv.gz"
# "NA" no Ames quer dizer "não tem" em várias colunas: só ele e a célula vazia contam como falta
df = pd.read_csv(AMES, keep_default_na=False, na_values=["NA", ""])

num_cols = ["OverallQual", "GrLivArea", "TotalBsmtSF", "LotArea", "GarageCars"]
cat_cols = ["Neighborhood", "HouseStyle", "ExterQual"]
X = df[num_cols + cat_cols]
y = df["SalePrice"].astype(float)

pre = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")),
                      ("sc",  StandardScaler())]), num_cols),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                      ("oh",  OneHotEncoder(handle_unknown="ignore"))]), cat_cols),
])
modelo = Pipeline([("pre", pre), ("m", Ridge())])
modelo.fit(X, y)
print(f"modelo treinado com {len(X)} casas")

modelo treinado com 1460 casas


A segunda célula salva **o pipeline inteiro** num arquivo: o preenchimento, a escala, a codificação e
o modelo. Ela também mostra a versão do scikit-learn que gerou o arquivo. **Anote essa versão**, porque
ela vai no `requirements.txt`.

In [2]:
import joblib
import sklearn

joblib.dump(modelo, "modelo_casas.pkl")
print("arquivo salvo: modelo_casas.pkl")
print(f"versão do scikit-learn: {sklearn.__version__}")    # a que vai no requirements.txt

arquivo salvo: modelo_casas.pkl
versão do scikit-learn: 1.6.1


A terceira célula baixa o arquivo do Colab para o seu computador. Ele vai para a pasta de downloads
do navegador.

In [3]:
try:
    from google.colab import files
    files.download("modelo_casas.pkl")        # o navegador baixa o arquivo
except ModuleNotFoundError:
    print("Fora do Colab: o modelo_casas.pkl está na mesma pasta deste notebook.")

Fora do Colab: o modelo_casas.pkl está na mesma pasta deste notebook.


Bad pipe message: %s [b' q=0.9, image/jxl, image/avif, image/webp, image/apng, */*; q=0.8, application/signed-exchange; v=b3; q=0.7', b'Host: localh']
Bad pipe message: %s [b't:41901\r\nUser-Agent: Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/1', b'.0.0.0 Safari/537.36\r\nAccept-Encoding: gzip, deflate', b'br, zstd\r\nAccept-Language: pt-BR, pt; q=0.9']
Bad pipe message: %s [b'en-US; q=0.8, en; q=0.7\r\nCache-Control: max']
Bad pipe message: %s [b'ge=0\r\nReferer: https://github.com/\r\nX-Reques', b'ID: d9a6674c-8580-4cfe-a9ee-3896f2802143\r\nX-Real-IP: 10.241.0.36\r\nX-Forwarded-Port: 443\r\nX-Forwarded-Scheme: https\r', b'-Original']
Bad pipe message: %s [b'RI: /\r\nX-Scheme: https\r\nsec-fetch-site: cros', b'site\r\nsec-fetch-mode: navigate\r\nsec-fetch-dest: document\r\nsec-ch-ua: "Not-A.Brand";v="99", "Google Chrome";v="157"', b'"Chromium";v="157"\r\nsec-ch-ua-mobile: ?0\r\ns']
Bad pipe message: %s [b' q=0.9, image/jxl, image/avif, image/

> 👀 **Se o download não começar**, o navegador pode ter bloqueado. Clique no ícone de **pasta**, na
> barra da esquerda do Colab, procure o `modelo_casas.pkl`, clique nos **três pontinhos** ao lado dele
> e em **Baixar**.

### Os outros três arquivos: a pasta `lab` do zip
O zip da aula, no EAD, traz este notebook e uma pasta `lab`, com os arquivos do app. Descompacte o
zip e mova o `modelo_casas.pkl` que você acabou de baixar para dentro da pasta `lab`. São **quatro**
arquivos que vão para o GitHub:

| arquivo | de onde vem | o que é |
|---|---|---|
| `modelo_casas.pkl` | deste notebook | o pipeline treinado, salvo com `joblib.dump` |
| `app.py` | pasta `lab` | o programa do app: carrega o modelo, mostra o formulário e calcula o preço |
| `requirements.txt` | pasta `lab` | as bibliotecas que o Streamlit instala antes de rodar o app |
| `README.md` | pasta `lab` | a página de apresentação do repositório |

> ⚠️ Deixe os quatro arquivos **na pasta `lab`**, com estes nomes exatos. A pasta tem também o
> `app_completo.py`, que só entra no L4, se você quiser. Se o navegador baixou o `.pkl` com outro nome, como `modelo_casas (1).pkl`,
> renomeie para `modelo_casas.pkl`.

### Olhar o app
Abra o `app.py` num editor de texto qualquer (o Bloco de Notas serve). São cerca de 60 linhas. Procure três trechos:

1. **onde carrega o modelo**: a função `carregar_modelo()`, com `joblib.load("modelo_casas.pkl")`. O
   `@st.cache_resource` em cima dela faz o arquivo ser lido uma vez só, e não a cada clique;
2. **o formulário**: os `st.slider`, `st.number_input` e `st.selectbox`, um para cada coluna do
   modelo. As áreas são pedidas em m², e a linha do `PES2_POR_M2` converte para pés quadrados, que é
   a unidade do Ames;
3. **onde prevê**: o `pd.DataFrame` com **os mesmos nomes de coluna do treino** (`OverallQual`,
   `GrLivArea`, ...) e o `modelo.predict(casa)`. A linha chega **bruta**: quem preenche, escala e
   codifica é o pipeline, como na regra de ouro da a11 (`fit` só no treino, `transform` em todo o
   resto, inclusive na produção).

> 👀 Repare que o app **não treina nada**. Ele só carrega o arquivo e chama `predict`.

### Conferir o `requirements.txt`
Abra também o `requirements.txt`. Ele fixa duas versões: a do `streamlit`, com que o app foi testado,
e a do `scikit-learn`, que tem de ser **a mesma que a segunda célula mostrou**. Se o número for
outro, troque a linha `scikit-learn==...` pela versão que você anotou e salve o arquivo antes do L2.
Com outra versão, o `.pkl` pode não carregar.

---
## L2 · Criar o repositório no GitHub

### Criar o repositório
1. Entre em https://github.com com a sua conta.
2. No canto superior direito, clique no **+** e depois em **New repository**.
3. Preencha:
   - **Repository name**: `app-casas-seunome` (troque `seunome` pelo seu nome);
   - **Public**: o repositório precisa ser **público** para o deploy gratuito;
   - **Add a README file**: deixe **desmarcado** (o README já vem pronto nos arquivos da aula).
4. Clique em **Create repository**.

### Enviar os 4 arquivos pelo navegador
1. Na página do repositório vazio, clique no link **uploading an existing file** (ou, num repositório
   que já tem arquivos, em **Add file → Upload files**).
2. Abra a pasta `lab` e arraste os **4 arquivos** (`app.py`, `modelo_casas.pkl`, `requirements.txt`,
   `README.md`) para a página, ou clique em **choose your files**. Arraste os arquivos, e não a pasta:
   com a pasta, eles ficam dentro de `lab/` no repositório, e o Streamlit não os encontra.
3. Em **Commit changes**, escreva uma mensagem curta, por exemplo `primeira versão do app`.
4. Clique em **Commit changes**.

### ✅ Conferir
A página do repositório deve mostrar os 4 arquivos, todos na raiz (fora de qualquer pasta), e o README
aparece embaixo da lista:

```
app-casas-seunome/
├── README.md
├── app.py
├── modelo_casas.pkl
└── requirements.txt
```

**Anote a URL do repositório:** `https://github.com/__________/app-casas-seunome`

---
## L3 · Pôr o app no ar no Streamlit Community Cloud

### Entrar com o GitHub
1. Abra uma nova aba em https://share.streamlit.io
2. Clique em **Continue with GitHub** e autorize o Streamlit a ler os seus repositórios, se ele pedir.

### Criar o app
1. No canto superior direito, clique em **Create app**.
2. Na pergunta *"Do you already have an app?"*, clique em **Yup, I have an app**.
3. Preencha:
   - **Repository**: `seu-usuario/app-casas-seunome`;
   - **Branch**: `main`;
   - **Main file path**: `app.py`;
   - **App URL** (opcional): um endereço mais fácil de lembrar, como `casas-seunome`.
4. Clique em **Deploy**.

> 🐍 A versão do Python fica em **Advanced settings**. O padrão do Streamlit (3.12) funciona com o
> scikit-learn 1.6.1; não precisa mudar.

### Acompanhar o build
O Streamlit agora faz, sozinho, o que você faria no seu computador:
1. copia o repositório do GitHub;
2. instala as bibliotecas do `requirements.txt`;
3. roda o `app.py`.

A tela mostra o andamento. Na primeira vez, o build leva alguns minutos.

### Testar
Quando o app abrir, você verá o título **🏠 Quanto vale esta casa?** e o formulário. Com os valores
que já vêm preenchidos, o preço previsto é **US$ 170.845**. Mude a qualidade, a área ou o bairro e
veja o preço mudar.

> 👀 Escolha o bairro **NridgHt (Northridge Heights)** e depois **IDOTRR (Iowa DOT and Rail Road)**,
> mantendo o resto igual. O modelo aprendeu com as vendas reais quanto o bairro pesa no preço.

**Anote a URL do seu app:** `https://__________.streamlit.app`

---
## L4 · Auto-deploy: mudar no GitHub e ver o app se refazer

O Streamlit **vigia o repositório**: cada mudança que você grava (um *commit*) no GitHub é copiada
para o app, sem você clicar em nada no Streamlit. Este é o exemplo do princípio de **automação** do
MLOps; na área, esse processo se chama **CI/CD** (integração e entrega contínuas).

1. No GitHub, abra o repositório e clique em `app.py`.
2. Clique no **lápis** (*Edit this file*), no canto superior direito do arquivo.
3. Procure a linha

   ```python
   st.caption("App publicado por: SEU NOME")
   ```

   e troque `SEU NOME` pelo seu nome. Se quiser, mude também o texto do `st.title`.
4. Clique em **Commit changes...** e, na janela que abre, em **Commit changes** de novo.
5. Volte à aba do app. Em poucos segundos ele se refaz com o seu nome; se não mudar, recarregue a
   página do navegador.

> 👀 Repare no que **não** aconteceu: você não subiu nada no Streamlit, não reinstalou biblioteca e
> não reiniciou o app. A mudança no GitHub bastou.

Para deixar o repositório completo, edite também o `README.md` do mesmo jeito e troque o
`https://SEU-APP.streamlit.app` pela URL do seu app.

### (Opcional) Uma mudança maior: o app completo
O `app.py` é simples de propósito, para ser lido inteiro. No EAD da disciplina há também o
`app_completo.py`, que faz **a mesma previsão**, com o mesmo modelo e os mesmos campos, numa tela com
mais recursos do Streamlit: o formulário na barra lateral, abas, o preço comparado com a casa típica,
um gráfico do preço conforme a área, a linha que o modelo recebe e o histórico das casas simuladas,
que se baixa em CSV.

1. Abra o `app_completo.py`, que está na pasta `lab` do zip, num editor de texto. Copie todo o conteúdo.
2. No GitHub, abra o `app.py`, clique no **lápis**, apague todo o conteúdo e cole o que copiou.
3. Troque `SEU NOME` pelo seu nome, de novo, e faça o *commit* como no passo 4.
4. Volte à aba do app e veja a tela nova.

> 👀 O `requirements.txt` não mudou: tudo o que a tela nova usa já vem com o Streamlit e o pandas.
> Se o app novo usasse outra biblioteca, ela teria de entrar no `requirements.txt`, ou o app
> pararia com o primeiro erro da tabela do L5.

> 💡 Na aba **E se a área mudar?**, cada ponto do gráfico é uma chamada ao mesmo `predict`, com a
> área trocada e o resto da casa fixo. É a mesma conta que dá o preço da sua casa, repetida para cada área.

Para voltar ao app simples, cole de novo o conteúdo do `app.py` da pasta `lab` e faça o *commit*.

---
## L5 · Quando dá errado (consulta)

**Onde ler o erro:** com o app aberto, clique em **Manage app**, no canto inferior direito. O painel
que abre mostra os **logs**: tudo o que o Streamlit escreveu ao instalar as bibliotecas e rodar o
`app.py`, com a mensagem de erro no fim. Leia a última mensagem antes de mexer em qualquer coisa.

| sintoma | causa provável | correção |
|---|---|---|
| `ModuleNotFoundError: No module named 'sklearn'` (ou outro nome) | falta a biblioteca no `requirements.txt` | acrescente a linha no `requirements.txt` pelo GitHub (o pacote se chama `scikit-learn`, e o módulo, `sklearn`) e faça o *commit* |
| `FileNotFoundError: ... 'modelo_casas.pkl'` | o arquivo não subiu, subiu dentro de uma pasta ou com outro nome | confira no GitHub se o `.pkl` está na raiz, com o nome exato `modelo_casas.pkl` |
| `InconsistentVersionWarning` nos logs, ou um erro ao carregar o `.pkl` | o scikit-learn instalado não é a versão que gerou o arquivo | deixe no `requirements.txt` a versão que a segunda célula do L1 mostrou |
| o app não mostra a mudança que você fez | o *commit* não foi feito, ou foi em outro ramo (*branch*) | confira no GitHub se a mudança está no `main`; em **Manage app**, use **Reboot app** |
| o repositório não aparece na lista do Streamlit | o repositório é privado, ou o Streamlit não tem permissão | deixe o repositório público, ou cole a URL do `app.py` em **Paste GitHub URL** |
| o app avisa que está dormindo | o app ficou 12 horas sem acesso | siga a tela para acordá-lo e espere um pouco |

> 📏 **Limites do plano gratuito** (conferidos na documentação do GitHub e do Streamlit em 02/11/2026;
> eles mudam com o tempo):
> - **Envio pelo navegador do GitHub:** até **25 MiB** por arquivo. Acima de 100 MiB, o GitHub bloqueia
>   o arquivo mesmo pela linha de comando. O `modelo_casas.pkl` tem 5 KB.
> - **Memória do app:** a documentação do Streamlit dá uma faixa aproximada, de 690 MB a 2,7 GB (valores de
>   fevereiro de 2024), e avisa que os limites podem mudar sem aviso. Um modelo grande ou um dataset
>   inteiro carregado no app pode estourar.
> - **O app dorme** depois de **12 horas sem acesso**. Antes de apresentar, abra o app e espere ele
>   acordar.

---
## L6 · (Opcional) O modelo do seu projeto num app

> Este passo **não é obrigatório** na disciplina. Ele é para o grupo que quiser mostrar o modelo do
> projeto funcionando.

O caminho é o mesmo do app das casas, trocando três coisas:

1. **O arquivo do modelo.** No notebook do projeto, depois de treinar **o pipeline inteiro** (o
   preenchimento, a escala, a codificação e o modelo, como na a11), salve e baixe:

   ```python
   import joblib, sklearn
   joblib.dump(pipeline, "modelo_projeto.pkl")
   print(sklearn.__version__)          # a versão que vai no requirements.txt

   from google.colab import files
   files.download("modelo_projeto.pkl")
   ```

   As três células do L1 são um exemplo completo desse passo.
2. **O formulário.** Um campo para cada coluna que o pipeline recebe, e um `pd.DataFrame` com **os
   mesmos nomes de coluna** do `X` do treino. Use `st.number_input` ou `st.slider` para números e
   `st.selectbox` para categorias, com as categorias que existem no dado. Se a unidade do formulário
   for diferente da do treino (como os m² do app das casas), converta antes do `predict`.
3. **O `requirements.txt`.** Fixe a versão do scikit-learn que o `print` mostrou, e acrescente as
   outras bibliotecas que o app importa.

Se o dado do projeto tem informação pessoal, ele **não vai** para o repositório público: só o
`.pkl`, o `app.py`, o `requirements.txt` e o `README.md`.

---
## 📌 Anexo: os componentes do Streamlit mais usados

```python
import streamlit as st

# texto
st.title("Título")
st.header("Cabeçalho")
st.write("Texto, números, tabelas")
st.markdown("**negrito** e *itálico*")
st.caption("Texto pequeno, em cinza")

# entrada
st.number_input("Número", min_value=0, max_value=100, value=10)
st.slider("Valor", 0, 100, 50)
st.selectbox("Escolha", ["A", "B", "C"])
st.text_input("Digite")
st.checkbox("Marcar")
st.button("Calcular")
st.file_uploader("Enviar um arquivo")

# saída
st.metric("Preço previsto", "US$ 170.845")
st.success("Deu certo")
st.warning("Atenção")
st.error("Erro")
st.dataframe(df)

# gráficos
st.pyplot(fig)            # uma figura do Matplotlib
st.plotly_chart(fig)      # uma figura do Plotly

# organização da tela
col1, col2 = st.columns(2)
with col1:
    st.write("coluna da esquerda")
with st.sidebar:
    st.write("barra lateral")
aba1, aba2 = st.tabs(["Aba 1", "Aba 2"])

# carregar o modelo uma vez só
@st.cache_resource
def carregar_modelo():
    return joblib.load("modelo.pkl")
```

**Para continuar**
- Para ir além do Lab: Kalinowski, M.; Escovedo, T.; Villamizar, H.; Lopes, H. *Engenharia de Software
  para Ciência de Dados: um guia de boas práticas com ênfase na construção de sistemas de Machine
  Learning em Python*. Casa do Código, 2023. O capítulo 12 trata da implantação de modelos, e o
  capítulo 16, de DevOps e MLOps.
- Documentação do Streamlit: https://docs.streamlit.io (a seção *Deploy* trata do Community Cloud).
- Documentação do GitHub sobre enviar arquivos pelo navegador:
  https://docs.github.com/en/repositories/working-with-files/managing-files/adding-a-file-to-a-repository
- Os dados: De Cock, D. *Ames, Iowa: Alternative to the Boston Housing Data as an End of Semester
  Regression Project*. Journal of Statistics Education, 19(3), 2011.